In [19]:
class Pair:
    def __init__(self, key: int, val: str):
        self.key = key
        self.val = val

In [20]:
class HashMapChaining:
    def __init__(self):
        self.size = 0
        self.capacity = 4
        self.load_thres = 2.0 / 3.0
        self.extend_ratio = 2
        self.buckets = [[] for _ in range(self.capacity)]

    def hash(self, key):
        return key % self.capacity

    def load_factor(self):
        return self.size / self.capacity

    def get(self, key):
        index = self.hash(key)
        bucket = self.buckets[index]
        for pair in bucket:
            if pair.key == key:
                return pair.val
        return None

    def put(self, key, val):
        if self.load_factor() > self.load_thres:
            self.extend()
        index = self.hash(key)
        bucket = self.buckets[index]

        for pair in bucket:
            if pair.key == key:
                pair.val = val
                return

        pair = Pair(key, val)
        bucket.append(pair)
        self.size += 1

    def remove(self, key):
        index = self.hash(key)
        bucket = self.buckets[index]
        
        for pair in bucket:
            if pair.key == key:
                bucket.remove(pair)
                self.size -= 1
                break

    def extend(self):
        buckets = self.buckets
        self.capacity *= self.extend_ratio
        self.buckets = [[] for _ in range(self.capacity)]
        self.size = 0
        for bucket in buckets:
            for pair in bucket:
                self.put(pair.key, pair.val)

    def print(self):
        for bucket in self.buckets:
            res = []
            for pair in bucket:
                res.append(str(pair.key) + " -> " + pair.val)
            print(res)

In [21]:
hash_map = HashMapChaining()
hash_map.put(1, "A")
hash_map.put(2, "B")
hash_map.print()


[]
['1 -> A']
['2 -> B']
[]


In [22]:
hash_map.put(3, "C")
hash_map.put(9, "D")
hash_map.put(10, "E")
hash_map.print()

[]
['1 -> A', '9 -> D']
['2 -> B', '10 -> E']
['3 -> C']
[]
[]
[]
[]


In [23]:
print(hash_map.get(9))
print(hash_map.get(11))

D
None


In [24]:
hash_map.remove(10)
hash_map.print()

[]
['1 -> A', '9 -> D']
['2 -> B']
['3 -> C']
[]
[]
[]
[]


In [25]:
class HashMapOpenAddressing:
    def __init__(self):
        self.size = 0 
        self.capacity = 4  
        self.load_thres = 2.0 / 3.0  
        self.extend_ratio = 2  
        self.buckets = [None] * self.capacity
        self.TOMBSTONE = Pair(-1, "-1")

    def print(self):
        for pair in self.buckets:
            if pair is None:
                print("None")
            elif pair is self.TOMBSTONE:
                print("TOMBSTONE")
            else:
                print(pair.key, "->", pair.val)

    def hash(self, key):
        return key % self.capacity

    def load_factor(self):
        return self.size / self.capacity

    def find_bucket(self, key):
        index = self.hash(key)
        first_tombstone = -1
        while self.buckets[index] is not None:
            if self.buckets[index].key == key:
                if first_tombstone != -1:
                    self.buckets[first_tombstone] = self.buckets[index]
                    self.buckets[index] = self.TOMBSTONE
                    return first_tombstone 
                return index 
            if first_tombstone == -1 and self.buckets[index] is self.TOMBSTONE:
                first_tombstone = index
            index = (index + 1) % self.capacity
        return index if first_tombstone == -1 else first_tombstone

    def get(self, key):
        index = self.find_bucket(key)
        if self.buckets[index] not in [None, self.TOMBSTONE]:
            return self.buckets[index].val
        return None

    def put(self, key, val):
        if self.load_factor() > self.load_thres:
            self.extend()
        index = self.find_bucket(key)
        if self.buckets[index] not in [None, self.TOMBSTONE]:
            self.buckets[index].val = val
            return
        self.buckets[index] = Pair(key, val)
        self.size += 1

    def remove(self, key):
        index = self.find_bucket(key)
        if self.buckets[index] not in [None, self.TOMBSTONE]:
            self.buckets[index] = self.TOMBSTONE
            self.size -= 1

    def extend(self):
        buckets_tmp = self.buckets
        self.capacity *= self.extend_ratio
        self.buckets = [None] * self.capacity
        self.size = 0
        for pair in buckets_tmp:
            if pair not in [None, self.TOMBSTONE]:
                self.put(pair.key, pair.val)

In [28]:
hash_map = HashMapOpenAddressing()
hash_map.put(1, "A")
hash_map.put(2, "B")
hash_map.print()



None
1 -> A
2 -> B
None


In [29]:
hash_map.put(3, "C")
hash_map.put(9, "D")
hash_map.put(10, "E")
hash_map.print()

None
1 -> A
2 -> B
3 -> C
9 -> D
10 -> E
None
None


In [30]:
print(hash_map.get(9))
print(hash_map.get(11))

D
None


In [31]:
hash_map.remove(10)
hash_map.print()

None
1 -> A
2 -> B
3 -> C
9 -> D
TOMBSTONE
None
None


In [1]:
a = set([1, 2])
b = set([2, 3])
inter = a & b
union = a | b
minus = a - b
diff = a ^ b

print(a)
print(b)
print(f"inter: {inter}")
print(f"union: {union}")
print(f"minus: {minus}")
print(f"diff: {diff}")

{1, 2}
{2, 3}
inter: {2}
union: {1, 2, 3}
minus: {1}
diff: {1, 3}
